# AlphaPulse — Week 1: Data Acquisition & Integrity Check

**Project:** Infotact Solutions Project 2 (Portfolio Risk & Volatility Monitor)  
**Primary Source:** Twelve Data API (with `yfinance` fallback)  
**Targets:** Local Parquet/CSV + **MongoDB Compass** (`alphapulse.market_prices`) + **Hugging Face Hub**

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path when running inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import time

import polars as pl

from scripts.week1_ingest import (
    align_and_clean,
    load_asset_series,
    push_to_huggingface,
    run_integrity_checks,
    save_to_mongodb,
)
from src.config import FX_ASSET, PORTFOLIO_ASSETS, PROCESSED_DIR, get_settings

settings = get_settings()
print(f"Loaded settings | Mongo DB: {settings.mongo_db} | HF Repo: {settings.hf_repo_id}")
td_ok = bool(settings.twelve_data_api_key and "your_" not in settings.twelve_data_api_key)
print(f"Twelve Data key configured: {td_ok}")

## 1. Fetch Historical OHLCV Data (Twelve Data + Fallback)
Pulls 5 years (`1260` trading days) of corporate-action-adjusted daily bars for `USD/INR`, Indian NSE equities, Gold BeES, and `BTC/USD`.

In [ ]:
fx_df = load_asset_series(FX_ASSET, settings)
asset_frames = []
for asset in PORTFOLIO_ASSETS:
    df_asset = load_asset_series(asset, settings)
    asset_frames.append(df_asset)
    time.sleep(1.0)

print(f"Fetched FX ({fx_df.height} rows) + {len(asset_frames)} portfolio assets.")

## 2. INR Normalization, Forward-Fill Alignment & `float32` Optimization

In [ ]:
cleaned_df = align_and_clean(asset_frames, fx_df)
run_integrity_checks(cleaned_df)

print(cleaned_df.schema)
print(f"Estimated memory: {cleaned_df.estimated_size('mb'):.3f} MB")
cleaned_df.group_by("asset_id", "sector", "source").agg(
    pl.len().alias("rows"),
    pl.col("date").min().alias("start_date"),
    pl.col("date").max().alias("end_date"),
    pl.col("close").min().alias("min_inr"),
    pl.col("close").max().alias("max_inr"),
).sort("asset_id")

## 3. Corporate Action & Split Continuity Audit

In [ ]:
jumps = (
    cleaned_df.sort(["asset_id", "date"])
    .with_columns((pl.col("close") / pl.col("close").shift(1).over("asset_id") - 1.0).alias("daily_ret"))
    .filter(pl.col("daily_ret").abs() > 0.60)
)
print(f"Extreme single-day jumps (>60%): {jumps.height}")
if jumps.height > 0:
    display(jumps)

## 4. Persist Locally, Upsert to MongoDB & Push to Hugging Face

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
parquet_path = PROCESSED_DIR / "market_prices.parquet"
csv_path = PROCESSED_DIR / "market_prices.csv"

cleaned_df.write_parquet(parquet_path)
cleaned_df.write_csv(csv_path)
print(f"Saved local Parquet & CSV to {PROCESSED_DIR}")

save_to_mongodb(cleaned_df, settings)
push_to_huggingface(str(parquet_path), settings)